# Notebook 08 — Security at Runtime

**Premise:** Notebook 07 showed the audit chain — *forensic*
verification of what happened. This notebook shows the *active*
defenses: things that fire **at runtime** to prevent harm before
it lands in the audit log.

These are the controls federal auditors and lab CISOs ask about first:

1. **PII redaction at the LLM boundary** — sensitive data never
   leaves the agent in plaintext.
2. **Sandboxed code execution** — the agent can write and run
   Python without ever touching the host's filesystem or network.
3. **Tool allowlists** — the agent literally cannot call a tool
   that's not in its sandbox.
4. **PII-safe audit logging** — what gets written to logs is
   metadata + classification, never raw content.
5. **Token / cost budgets** — runaway loops trip a circuit breaker,
   not your finance department.

By the end you'll have run each one and seen its effect on the
wire — what the LLM saw vs. what the user wrote, what was denied
vs. allowed, and how each event landed in the audit chain.

## Setup

**Wire up the security toolkit.** One import block pulls in the model loader, the PII detector + redactor, the arcrun loop with its `SandboxConfig`, and a `rich` console for the tables we print throughout. Run it once.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from roadshow import connect, Message
from arcllm._pii import RegexPiiDetector, redact_text
from arcrun import run, SandboxConfig, make_execute_tool, Tool, StaticProvider
from rich import print
from rich.panel import Panel
from rich.console import Console
from rich.table import Table

console = Console()

## 1. PII detection — what's in the message?

ArcLLM ships a regex-based PII detector with built-in patterns for
SSN, credit card, email, IP address, and phone number. It's
pluggable — you can swap in a Presidio or Microsoft DLP backend
for higher-stakes deployments — but the regex backend is fine for
demos and most lab use cases.

🔧 **EDIT ME:** Replace `RAW` with text from your domain that has
the kind of identifiers you'd want to protect. Re-run the cell.
Anything the regex misses? That's signal — that's what a custom
pattern (next section) is for.

**Define the message and run the built-in detector.** `RegexPiiDetector` ships with patterns for SSN, credit card, email, IP, and phone — we point it at one sample note and collect every match.

🔧 **EDIT ME:** Replace `RAW` with text from your domain. Anything the regex misses is signal for the custom patterns in the next section.

In [ ]:
# 🔧 EDIT ME — paste real-feeling text from your domain (sanitize first).
RAW = (
    'Hi, please email jane.doe@nationallab.gov about subject 47. '
    'Her SSN is 123-45-6789 and the corp card on file is 4242 4242 4242 4242. '
    'My cell is 555-867-5309. The job ran on lab asset LAB-A4427 at 10.7.42.4.'
)

detector = RegexPiiDetector()
matches = detector.detect(RAW)

**Show what it found and what gets redacted.** The table lists each PII span; `redact_text` swaps every match for a `[PII:TYPE]` placeholder — that redacted form is what would leave the agent.

In [ ]:
tbl = Table(title='PII findings')
tbl.add_column('type', style='red')
tbl.add_column('span')
tbl.add_column('matched text')
for m in matches:
    tbl.add_row(m.pii_type, f'{m.start}-{m.end}', repr(m.matched_text))
console.print(tbl)

console.print(Panel(redact_text(RAW, matches), title='redacted', border_style='green'))

## 2. Add domain-specific regex patterns

`RegexPiiDetector` accepts `custom_patterns=` directly — pass a
list of `{'name': ..., 'pattern': ...}` dicts and the detector
compiles them alongside the built-ins. **No subclassing, no
monkey-patching.** This is how you bake your facility's
identifier scheme into the standard pipeline.

🔧 **EDIT ME:** The `MY_CUSTOM_PATTERNS` list below has three
common shapes (lab asset, project code, badge). Add or remove
entries to match *your* facility. Re-run and watch the new
types appear in the findings table.

**Add your facility's identifier shapes.** Pass a list of `{'name', 'pattern'}` dicts as `custom_patterns=` and they compile alongside the built-ins — no subclassing.

🔧 **EDIT ME:** Adjust `MY_CUSTOM_PATTERNS` to match *your* identifiers.

In [ ]:
# 🔧 EDIT ME — patterns specific to your facility.
MY_CUSTOM_PATTERNS = [
    {'name': 'LAB_ASSET',    'pattern': r'\bLAB-[A-Z]\d{4}\b'},        # e.g. LAB-A4427
    {'name': 'PROJECT_CODE', 'pattern': r'\bP-\d{2}-\d{4}\b'},           # e.g. P-26-0184
    {'name': 'BADGE_NUMBER', 'pattern': r'\bBN[A-Z]?\d{6}\b'},          # e.g. BN048291 / BNA048291
    {'name': 'INTERNAL_DOI', 'pattern': r'\bDOI:[a-z]+/[a-z0-9._-]+\b'},  # e.g. DOI:lab/internal-2026-009
]

extended_detector = RegexPiiDetector(custom_patterns=MY_CUSTOM_PATTERNS)

**A richer sample note** carrying both built-in PII and the custom shapes (badge, project code, lab asset, internal DOI).

In [ ]:
RAW_RICH = (
    'Hi, please email jane.doe@nationallab.gov about subject 47. '
    'Her SSN is 123-45-6789, badge BN048291, project P-26-0184. '
    'The job ran on lab asset LAB-A4427 at 10.7.42.4. '
    'Internal reference: DOI:lab/internal-2026-009.'
)

**Detect and redact.** Watch the new custom types appear in the findings table right next to the built-ins.

In [ ]:
matches = extended_detector.detect(RAW_RICH)
tbl = Table(title='PII findings (built-ins + your custom patterns)')
tbl.add_column('type', style='red'); tbl.add_column('span'); tbl.add_column('matched text')
for m in matches:
    tbl.add_row(m.pii_type, f'{m.start}-{m.end}', repr(m.matched_text))
console.print(tbl)
console.print(Panel(redact_text(RAW_RICH, matches), title='redacted', border_style='green'))

✅ **TODO:** Add a 5th pattern for one identifier shape *your*
team uses (Jira-style ticket id, ServiceNow ID, hostname
convention, etc.). Run again. The same `MY_CUSTOM_PATTERNS`
list is what you'd ship to the `SecurityModule` in production
— `connect(security={'pii_enabled': True,
'custom_patterns': MY_CUSTOM_PATTERNS})`.

## 2b. NER-based detection — a spaCy model for the things regex can't catch

Regex catches *shapes* (digits in a row, slash-separated paths).
It can't catch *named entities* — the model has to know that
*"Dr. Patel"* is a PERSON, *"Argonne"* is an ORG, *"Oak Ridge,
TN"* is a GPE (geo-political entity). For that we need NER.

spaCy ships a small English model (`en_core_web_sm`, ~13 MB)
that does this in <50 ms per sentence. Below: load the model,
wrap it as a detector that returns `PiiMatch` objects, and
*combine* it with the regex detector. **Layered defense.**

**Setup.** `spacy` is in the project's `pyproject.toml`, so
`uv sync` already installed it. The **model** is shipped as a
vendored directory at `data/models/en_core_web_sm-3.8.0/` (~15 MB
— committable, audit-friendly, runs zero-network).

If you ever need to re-fetch or upgrade the model:

```bash
# Option A — install as a Python package (lives in .venv, invisible to repo)
uv run python -m spacy download en_core_web_sm

# Option B — copy the package's model artifacts into the repo (what we did)
cp -R .venv/lib/python3.13/site-packages/en_core_web_sm/en_core_web_sm-3.8.0 \
      data/models/
```

Air-gapped labs: a copy of `data/models/en_core_web_sm-3.8.0/`
is all you need — no `github.com` access, no `pip` resolution.
Mirror this folder onto the air-gapped machine and the notebook
runs.

**Load the vendored spaCy model.** We load `en_core_web_sm` from the repo path first (zero-network, air-gap friendly) and fall back to the installed package, setting `SPACY_OK` so later cells degrade gracefully if it's missing.

In [ ]:
import spacy
from pathlib import Path

# Vendored model in the repo. Loading from a path always works as long as
# data/models/<model>/ has meta.json + config.cfg + pipe folders.
MODEL_PATH = Path('../data/models/en_core_web_sm-3.8.0').resolve()

try:
    if MODEL_PATH.exists():
        nlp = spacy.load(str(MODEL_PATH))
        source = f'project path: {MODEL_PATH.relative_to(MODEL_PATH.parent.parent.parent)}'
    else:
        # Fallback: the model installed as a package (less portable, but works).
        nlp = spacy.load('en_core_web_sm')
        source = 'package: site-packages/en_core_web_sm'
    SPACY_OK = True
    print(f'spaCy {spacy.__version__} ready · {source}')
    print(f'  pipes:  {nlp.pipe_names}')
    print(f'  meta:   {nlp.meta["name"]} v{nlp.meta["version"]} ({nlp.meta["lang"]})')
except Exception as e:
    SPACY_OK = False
    print(f'Could not load spaCy model: {type(e).__name__}: {e}')
    print('Re-fetch with:  uv run python -m spacy download en_core_web_sm')

**Map spaCy's entity labels to our PII types.** NER tags like PERSON, ORG, GPE don't have a regex shape — this dict decides which of them your policy treats as PII.

In [ ]:
from arcllm._pii import PiiMatch

# Map spaCy NER labels → our PII types. en_core_web_sm tags include:
#   PERSON, NORP (nationalities/religions), ORG, GPE (city/country),
#   LOC, FAC (facility), DATE, MONEY, EVENT, PRODUCT, LAW, LANGUAGE.
# Pick the ones your policy considers PII for your domain.
SPACY_PII_LABELS = {
    'PERSON':   'PERSON_NAME',
    'ORG':      'ORGANIZATION',
    'GPE':      'LOCATION',
    'LOC':      'LOCATION',
    'FAC':      'FACILITY',
    'NORP':     'NATIONALITY_OR_RELIGION',
}

**Wrap the model as a detector.** `spacy_detect` runs NER and returns the same `PiiMatch` objects the regex detector produces — so the two are composable.

In [ ]:
def spacy_detect(text: str) -> list[PiiMatch]:
    """Run en_core_web_sm NER and return PiiMatch objects.
    Same shape as RegexPiiDetector.detect() — composable."""
    if not SPACY_OK:
        return []
    doc = nlp(text)
    out = []
    for ent in doc.ents:
        pii_type = SPACY_PII_LABELS.get(ent.label_)
        if pii_type:
            out.append(PiiMatch(pii_type=pii_type, matched_text=ent.text,
                                start=ent.start_char, end=ent.end_char))
    return out

**A sentence full of named entities** — people, an org, a place — the kind of PII only NER can catch.

In [ ]:
RAW_NER = (
    'Dr. Aniya Patel from Argonne National Laboratory met with '
    'Marcus Reyes (Oak Ridge, TN) yesterday to discuss the '
    'Frontier procurement. Contact: aniya.patel@anl.gov, '
    'badge BN048291, asset LAB-A4427.'
)

**Run NER and show the findings.** These are the entities regex would have missed entirely.

In [ ]:
if SPACY_OK:
    ner_matches = spacy_detect(RAW_NER)
    tbl = Table(title='spaCy NER findings')
    tbl.add_column('type', style='magenta'); tbl.add_column('span'); tbl.add_column('matched text')
    for m in ner_matches:
        tbl.add_row(m.pii_type, f'{m.start}-{m.end}', repr(m.matched_text))
    console.print(tbl)
else:
    print('(spaCy section skipped — install per the cell above)')

## 2c. Layered detector — regex + NER, combined

Each detector catches what the other misses. Regex is fast and
deterministic for known shapes (SSN, lab assets). NER catches
named entities. Run both, merge results, redact once. This is
what production deployments look like.

**Combine both detectors.** `layered_detect` runs regex + NER, then dedupes overlapping spans — each layer catches what the other misses.

In [ ]:
def layered_detect(text: str) -> list[PiiMatch]:
    """Run regex + NER, dedupe overlapping spans (NER usually wins on conflict)."""
    regex_hits = extended_detector.detect(text)
    ner_hits   = spacy_detect(text)
    all_hits   = regex_hits + ner_hits
    # Dedupe by (start, end) — a match at the same span gets kept once.
    seen = {}
    for m in sorted(all_hits, key=lambda h: (h.start, -len(h.matched_text))):
        key = (m.start, m.end)
        seen.setdefault(key, m)
    return list(seen.values())

**Run the layered detector and redact once.** The `source` column shows which layer caught each hit; the panel is the final redacted text.

In [ ]:
all_hits = layered_detect(RAW_NER)
tbl = Table(title='Layered detector — regex + NER')
tbl.add_column('source', style='dim')
tbl.add_column('type'); tbl.add_column('span'); tbl.add_column('matched text')
regex_types = {p['name'] for p in MY_CUSTOM_PATTERNS} | {'EMAIL', 'SSN', 'IPV4', 'PHONE', 'CREDIT_CARD'}
for m in all_hits:
    src = 'regex' if m.pii_type in regex_types else 'spacy NER'
    color = 'red' if src == 'regex' else 'magenta'
    tbl.add_row(src, f'[{color}]{m.pii_type}[/{color}]', f'{m.start}-{m.end}', repr(m.matched_text))
console.print(tbl)
console.print(Panel(redact_text(RAW_NER, all_hits), title='final redacted text', border_style='green'))

Two things to notice:

1. The **regex** layer catches the email, badge, and lab asset
   — high-precision, zero-cost lookups for known shapes.
2. The **NER** layer catches `Dr. Aniya Patel`, `Argonne
   National Laboratory`, `Marcus Reyes`, `Oak Ridge`, `TN` —
   things that don't have a regex shape but are absolutely PII.

**Production note.** spaCy's `en_core_web_sm` is fast but
only ~80% F1 on PERSON. For higher-stakes use cases swap in:

- `en_core_web_trf` (transformer-backed, ~95% F1, ~500 MB,
  GPU-friendly)
- **Microsoft Presidio** (regex + spaCy + ML scorers + custom
  recognizers, plug-in compatible with most pipelines)
- An on-prem Llama / Phi model behind a thin FastAPI
  classifier (full air-gap, full audit)

The interface is identical — return `PiiMatch` objects. Swap
the implementation, keep the rest of the harness untouched.

## 3. Plug it into the LLM call — `SecurityModule`

Setting `security={'pii_enabled': True}` on `connect()` wraps
the adapter so **every outbound message is redacted before the
network call**, and **every inbound response is scanned on the
way back**. The model literally cannot see the raw SSN.

Watch the response — the model tells us it sees `[PII:SSN]`
instead of the actual number.

**Load a model with redaction on, then call it.** With `security={'pii_enabled': True}` every outbound message is redacted before the network call — the model sees `[PII:SSN]`, never the real number.

In [ ]:
model_redacted = connect(security={'pii_enabled': True})

resp = await model_redacted.invoke([
    Message(role='user', content=f'Summarize what you can see about this customer: {RAW}')
])
console.print(Panel(resp.content or '', title='LLM response (saw redacted text)', border_style='cyan'))

## 3a. Toggle redaction live — see the difference on the wire

Toggle the switch and rerun. With redaction OFF, the model sees
the raw SSN. With it ON, the model never sees it. Same prompt,
same model — only the security wrap changes.

**Build a live toggle.** This wires up an ipywidgets panel so you can flip redaction on/off and resend the same message — watch the model's reply change based only on the security wrap.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import asyncio

redact_toggle = widgets.ToggleButton(value=True, description='PII redaction',
                                     button_style='success',
                                     icon='shield')
go = widgets.Button(description='Send ▶', button_style='primary')
out = widgets.Output()
msg_in = widgets.Textarea(value=RAW, layout=widgets.Layout(width='100%', height='90px'),
                          description='Message:', style={'description_width': 'initial'})


async def _run_send():
    with out:
        out.clear_output(wait=True)
        console.print(f'[dim]sending… (redaction={redact_toggle.value})[/dim]')
    try:
        m = connect(security={'pii_enabled': bool(redact_toggle.value)})
        r = await m.invoke([Message(role='user', content=f'Summarize what you see in this customer note: {msg_in.value}')])
    except Exception as exc:
        with out:
            out.clear_output(wait=True)
            console.print(f'[red]error:[/red] {type(exc).__name__}: {exc}')
        return
    with out:
        out.clear_output(wait=True)
        title = f'redaction = {redact_toggle.value}'
        console.print(Panel(r.content or '', title=title, border_style='green' if redact_toggle.value else 'red'))


def _on_click_send(_b):
    task = asyncio.ensure_future(_run_send())
    def _check(t):
        if t.exception() is not None:
            with out:
                console.print(f'[red]task error:[/red] {t.exception()!r}')
    task.add_done_callback(_check)


go.on_click(_on_click_send)
display(widgets.VBox([msg_in, widgets.HBox([redact_toggle, go]), out]))


**This is the lethal-trifecta break.** Private data + external
comms + untrusted input is OWASP LLM02 (Sensitive Information
Disclosure). Redacting before egress severs it.

In federal mode the SecurityModule also signs the request payload
(Ed25519) so the receiving end can verify what was sent.

## 3. Sandbox — only allowed tools fire

Now the loop side. `SandboxConfig(allowed_tools=[...])` is a hard
permission boundary checked **before every tool dispatch**. The
model can ask for any tool; the sandbox will deny anything not on
the list and emit a `tool.denied` event you can audit.

Build two scenarios with the *same* tool — once in the allowlist,
once not. Watch the difference.

**Scenario A — tool on the allowlist.** Run the loop with `SandboxConfig(allowed_tools=['execute_python'])`; the tool is permitted, so no `tool.denied` events fire.

In [ ]:
model = connect()
exec_tool = make_execute_tool(timeout_seconds=10, max_output_bytes=4096)

events_open: list = []
result_open = await run(
    model=model, capabilities=StaticProvider([exec_tool]),
    sandbox=SandboxConfig(allowed_tools=['execute_python']),
    system_prompt='Use execute_python when math is needed.',
    task='Compute the first 10 Fibonacci numbers and print them.',
    on_event=events_open.append,
)
console.print(Panel(result_open.content or '', title='OPEN sandbox — tool allowed', border_style='green'))
n_denied = sum(1 for e in events_open if e.type == 'tool.denied')
print(f'  turns={result_open.turns}  tool_calls={result_open.tool_calls_made}  tool.denied events: {n_denied}')

**Scenario B — empty allowlist.** Same tool, same task, but `allowed_tools=[]` — the sandbox denies the call before dispatch and emits a `tool.denied` event you can audit.

In [ ]:
events_closed: list = []
result_closed = await run(
    model=model, capabilities=StaticProvider([exec_tool]),
    sandbox=SandboxConfig(allowed_tools=[]),  # empty allowlist => nothing fires
    system_prompt='Use execute_python when math is needed.',
    task='Compute the first 5 Fibonacci numbers.',
    on_event=events_closed.append,
    max_turns=3,
)
denied = [dict(e.data) for e in events_closed if e.type == 'tool.denied']
console.print(Panel(str(denied), title='CLOSED sandbox — tool.denied events', border_style='red'))
print(f'  turns={result_closed.turns}  tool_calls={result_closed.tool_calls_made}')

Same tool. Same task. Same model. The sandbox stops it cold. And
every denial is in the audit chain (verifiable end-to-end with
`result.verify_integrity()` from notebook 07) — so a SOC analyst
can prove that an attempted action was blocked.

🔧 **Try it (no answer cell):** Build a sandbox that allows ONLY
`read_file` (no execute, no list). Run a task that *needs* both
("Compute file sizes for every notebook"). What happens?
Does the agent retry, give up, or work around it? The trace will
tell you.

**Scratchpad — your read-only sandbox.** A `read_file` tool is defined for you; uncomment the run block to try a task that needs more than reading and watch the trace.

In [ ]:
# 🔧 SCRATCHPAD — read-only sandbox.
from pathlib import Path as _Path

async def _read(args, ctx):
    p = _Path(args['path']).expanduser()
    return p.read_text()[:500] if p.exists() else 'no'

read_only_tool = Tool(name='read_file', description='Read a file.',
                      input_schema={'type':'object','properties':{'path':{'type':'string'}},'required':['path']},
                      execute=_read)

# scenario_events = []
# r = await run(
#     model=model, capabilities=StaticProvider([read_only_tool, exec_tool]),
#     sandbox=SandboxConfig(allowed_tools=['read_file']),  # only read allowed
#     system_prompt='Use any tool to compute file sizes for the course notebooks.',
#     task='Compute total bytes across all notebooks in ./notebooks/.',
#     on_event=scenario_events.append, max_turns=4,
# )
# print('denied:', sum(1 for e in scenario_events if e.type == 'tool.denied'))
# print(r.content)

## 3b. Tool signing — provenance for every tool the agent loads

An allowlist tells the loop *which tools may run*. Tool **signing**
tells the loop *which tools were authored by someone we trust*.
Same Ed25519 primitives as `arctrust` — every tool bundle is
signed by an issuer DID; the loader verifies the signature before
registering the tool.

The teaching version: hash the tool's bytes, sign with a private
key, store the signature next to the tool. Loader recomputes the
hash, verifies against the issuer's public key. **If the bytes
have been modified after signing, verification fails — the tool
won't load.**

Production version (`arcskill` + `arctrust`): full bundle format
with manifest, dependencies, classification level, and per-
operator trust store. Same shape; more rigor.

**Mint the issuer keypair.** This Ed25519 keypair represents the lab's tool-publishing authority; its DID is how we'll identify who signed a tool.

In [ ]:
import hashlib
from arctrust import generate_keypair, sign, verify

# An issuer keypair — represents the lab's tool-publishing authority.
issuer = generate_keypair()
issuer_did = f'did:arc:lab-tools-{issuer.public_key[:4].hex()}'
print(f'Issuer DID: {issuer_did}')
print(f'Issuer pubkey:  {issuer.public_key.hex()[:32]}…')

**Sign a tool bundle.** Hash the tool's source bytes and sign the digest — the signature plus hash plus issuer DID make up the envelope stored next to the tool.

In [ ]:
def sign_tool_bundle(tool_source: bytes, *, issuer_priv: bytes) -> dict:
    """Hash + sign a tool's source bytes. Returns the bundle envelope."""
    digest = hashlib.sha256(tool_source).digest()
    signature = sign(digest, issuer_priv)
    return {
        'sha256':    digest.hex(),
        'signature': signature.hex(),
        'issuer':    issuer_did,
    }

**Verify a tool bundle.** Recompute the hash, confirm it matches, then check the signature against a trusted issuer's public key — fail closed on any mismatch.

In [ ]:
def verify_tool_bundle(tool_source: bytes, envelope: dict, *, trust_store: dict[str, bytes]) -> bool:
    """Verify a tool bundle against a trust store mapping issuer DID → public key."""
    expected_digest = hashlib.sha256(tool_source).digest()
    if expected_digest.hex() != envelope['sha256']:
        return False  # source changed since signing
    pubkey = trust_store.get(envelope['issuer'])
    if pubkey is None:
        return False  # issuer not trusted
    return verify(expected_digest, bytes.fromhex(envelope['signature']), pubkey)

**Build the trust store and sign a real tool.** We register our issuer, grab the source bytes of `make_execute_tool`, and produce its signed envelope.

In [ ]:
TRUSTED_ISSUERS = {issuer_did: issuer.public_key}

import inspect
from arcrun import make_execute_tool as _make_exec
tool_source = inspect.getsource(_make_exec).encode()
envelope = sign_tool_bundle(tool_source, issuer_priv=issuer.private_key)

console.print(Panel(
    f'tool source bytes: {len(tool_source):,}\n'
    f'sha256:    {envelope["sha256"][:32]}…\n'
    f'signature: {envelope["signature"][:32]}…\n'
    f'issuer:    {envelope["issuer"]}',
    title='signed tool envelope', border_style='cyan'))

**Try to break it three ways.** Verify the clean source (passes), then tampered source (hash mismatch) and an empty trust store (untrusted issuer) — both fail.

In [ ]:
ok = verify_tool_bundle(tool_source, envelope, trust_store=TRUSTED_ISSUERS)
print(f'\nverification (clean source):       {ok}')

tampered = tool_source + b'\n# malicious injection'
ok = verify_tool_bundle(tampered, envelope, trust_store=TRUSTED_ISSUERS)
print(f'verification (tampered source):    {ok}')

ok = verify_tool_bundle(tool_source, envelope, trust_store={})
print(f'verification (issuer not trusted): {ok}')

Three failure modes — all caught:

1. **Source byte tampering** — sha256 mismatch.
2. **Signature forgery** — Ed25519 verify fails (60-byte forgery
   probability ≈ 2⁻¹²⁸, computationally infeasible).
3. **Untrusted issuer** — DID isn't in the trust store, refuse to
   load even if the math checks out.

✅ **TODO:** Generate a *second* issuer keypair (representing a
different vendor or a malicious actor). Sign the same tool source
with the new key. Watch verification fail because the new
issuer's DID isn't in `TRUSTED_ISSUERS`.

## 3c. Policy-based authorization — per-call allow/deny

`SandboxConfig.allowed_tools` is a coarse boundary ("this tool
may run at all"). The **policy engine** in `arcagent` is the
fine-grained one: every individual tool *call* is evaluated,
with the agent's DID, classification, arguments, and environment
context fed into a multi-layer pipeline. First-deny-wins, fail-
closed.

The teaching version below is a 30-line policy engine that
authorizes each call against a list of rules. The production
version (`arcagent.modules.tool_policy_layers`) layers Global
→ Provider → Agent → Team → Sandbox layers and emits a
`policy.decision` event into the audit chain (NB07).

**Define the policy records.** `PolicyDecision` carries the outcome + rule + reason; `PolicyContext` carries the request's identity, classification, and time-of-day — the inputs every rule reasons over.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PolicyDecision:
    outcome: str   # 'allow' or 'deny'
    rule_id: str
    reason:  str

@dataclass
class PolicyContext:
    agent_did:      str
    classification: str    # 'unclassified' | 'cui' | 'secret'
    business_hours: bool

**The policy engine itself.** Four layers — classification, time-of-day, agent identity, argument inspection — evaluated first-deny-wins, defaulting to allow.

In [ ]:
def evaluate_policy(tool_name: str, arguments: dict, ctx: PolicyContext) -> PolicyDecision:
    """Tiny policy engine — first-deny-wins. Real systems evaluate dozens of layers."""
    # Layer 1: classification
    if tool_name == 'execute_python' and ctx.classification == 'secret':
        return PolicyDecision('deny', 'CLS-001', 'execute_python forbidden at SECRET classification')
    # Layer 2: time-of-day
    if tool_name == 'rm' and not ctx.business_hours:
        return PolicyDecision('deny', 'TIME-002', 'rm only allowed during business hours')
    # Layer 3: agent identity
    if tool_name == 'modify_run' and not ctx.agent_did.startswith('did:arc:ops-'):
        return PolicyDecision('deny', 'IDENT-003', f'modify_run restricted to ops agents; saw {ctx.agent_did}')
    # Layer 4: argument inspection
    if tool_name == 'execute_python' and 'subprocess' in arguments.get('code', ''):
        return PolicyDecision('deny', 'CONTENT-004', 'subprocess invocation forbidden in agent-authored code')
    return PolicyDecision('allow', 'DEFAULT', 'no rule matched — default-allow')

**A fleet of test calls** spanning each rule: the same call allowed for one context and denied for another.

In [ ]:
# Evaluate a fleet of tool calls against the policy.
test_cases = [
    ('execute_python', {'code': 'print(1+1)'}, PolicyContext('did:arc:research-007', 'unclassified', True)),
    ('execute_python', {'code': 'print(1+1)'}, PolicyContext('did:arc:research-007', 'secret',       True)),
    ('execute_python', {'code': 'import subprocess; subprocess.run(["ls"])'}, PolicyContext('did:arc:research-007', 'unclassified', True)),
    ('rm',             {'path': '/scratch/foo'}, PolicyContext('did:arc:research-007', 'unclassified', False)),
    ('modify_run',     {'run_id': '42'},          PolicyContext('did:arc:research-007', 'unclassified', True)),
    ('modify_run',     {'run_id': '42'},          PolicyContext('did:arc:ops-014',     'unclassified', True)),
]

**Evaluate every call and show the verdicts.** Each row is one decision — in production this becomes a signed `policy.decision` event in the audit chain.

In [ ]:
tbl = Table(title='Policy decisions (first-deny-wins)')
tbl.add_column('tool'); tbl.add_column('agent DID'); tbl.add_column('outcome', style='bold')
tbl.add_column('rule'); tbl.add_column('reason')
for tool_name, args, ctx in test_cases:
    d = evaluate_policy(tool_name, args, ctx)
    color = 'green' if d.outcome == 'allow' else 'red'
    tbl.add_row(tool_name, ctx.agent_did, f'[{color}]{d.outcome}[/{color}]', d.rule_id, d.reason)
console.print(tbl)

Each decision in the table would, in production, become a
`policy.decision` event in the audit chain — DID + tool name +
outcome + rule_id + reason, signed with the policy engine's own
DID. Months later an auditor can prove that *this exact tool
call* was *denied by this exact rule* at *this exact time*.

Two production layers go beyond what we showed:

- **Forbidden compositions.** *"Any agent that has access to
  PII redaction must NOT also have execute_python"* — prevents
  prompt-injected exfiltration.
- **Bundle-age policy.** *"Reject calls if the policy bundle
  hasn't refreshed in 24h"* — limits stale-rule blast radius.

✅ **TODO:** Add a 5th layer to `evaluate_policy` that denies
any `read_file` call where the path starts with `/etc/`. Run
the test cases again with one new entry that should be denied.

## 4. Sandboxed code execution — `make_execute_tool`

Look at the protections layered into the local subprocess sandbox:

- **Process isolation**: new process group (`start_new_session=True`)
  — the agent can't signal the host process.
- **Locked PATH**: only `/usr/bin:/bin` — no `pip install foo`.
- **Hard timeout**: SIGTERM at the deadline, SIGKILL after a 5s
  grace — runaway scripts can't hang the loop.
- **Output cap**: stdout+stderr truncated at `max_output_bytes` so
  a fork-bomb of `print('x')` can't OOM the audit log.
- **TempDir cwd**: each invocation gets a fresh
  `tempfile.TemporaryDirectory` — no cross-invocation FS state.

Demo it — give the agent a small computational task and watch what
code it writes.

**Run a real computation in the sandbox.** Give the agent `execute_python` and a prime-sieve task; we capture the loop's events so we can print the exact code it wrote and its final answer.

In [ ]:
events_exec: list = []
exec_result = await run(
    model=model, capabilities=StaticProvider([make_execute_tool(timeout_seconds=15, max_output_bytes=8192)]),
    sandbox=SandboxConfig(allowed_tools=['execute_python']),
    system_prompt='Use execute_python to compute things rather than reasoning step-by-step.',
    task='Find all prime numbers under 100 using a sieve. Print them comma-separated.',
    on_event=events_exec.append,
)

for e in events_exec:
    if e.type == 'tool.start' and e.data.get('name') == 'execute_python':
        code_run = e.data.get('arguments', {}).get('code', '')
        console.print(Panel(code_run, title='code the agent wrote', border_style='yellow'))

console.print(Panel(exec_result.content or '', title='final answer', border_style='cyan'))

## 5. Container isolation for higher tiers — `make_contained_execute_tool`

For air-gapped, federal, or anything where subprocess isn't enough,
arcrun also ships `make_contained_execute_tool` — Docker/Podman
container per execution. Same interface as the local tool, but the
code now runs inside a container with explicit limits.

We won't run it here (requires Docker on the demo machine), but
the constructor surface tells you what's enforced:

```python
from arcrun.builtins.contained_execute import make_contained_execute_tool
tool = make_contained_execute_tool(
    image='python:3.12-slim',
    timeout_seconds=30,
    max_output_bytes=65536,
    mem_limit='256m',         # cgroup memory ceiling
    cpu_period=100_000,       # CFS period (microseconds)
    cpu_quota=50_000,         # 50% of one core
    pids_limit=64,            # process count cap
    tmpfs_size='64m',         # writable tmpfs only — no host FS
    network_disabled=True,    # no egress, no exfiltration
)
```

OOM kill becomes `SandboxOOMError`, exceeded timeout becomes
`SandboxTimeoutError`, container failures become
`SandboxRuntimeError`, missing runtime is `SandboxUnavailableError`.
Each is auditable; each is recoverable.

**Federal posture progression:** subprocess sandbox for
personal/dev → container sandbox for enterprise/SCIF → Firecracker
microVM (planned) for the strictest tiers. Same agent code; the
sandbox swap is a config flag.

## 6. PII-safe audit by default — `AuditModule`

ArcLLM's `AuditModule` logs **metadata only** by default — provider,
model, message count, stop reason, content length, tool counts. It
does **not** log message content unless you explicitly opt in via
`include_messages=True` (DEBUG level) — and even then, ideally only
after redaction (i.e. after the SecurityModule has run).

Module stack order matters: **Audit → Security → Adapter**. Audit
sees what Security has already redacted. There is no path where
raw PII reaches the audit sink.

**Compose audit + security and make one call.** The module stack runs Audit → Security → Adapter, so audit only ever sees redacted content; the log line below carries metadata only, never message text.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s %(message)s', force=True)

model_audited = connect(
    security={'pii_enabled': True},
    audit={'log_level': 'INFO'},
)
_ = await model_audited.invoke([Message(role='user', content='What is 2 + 2?')])
print('(audit metadata logged above by arcllm.modules.audit — no message content)')

## 7. Budgets — `task_complete` and `make_budget_breach_args`

Token and cost budgets are part of the loop, not the application
code. Set them once and the loop terminates cleanly when crossed,
emitting a `task_complete` with `status='budget_breach'` so the
auditor can see *why* the run stopped.

ArcRun ships a `make_task_complete_tool` factory — pair it with
`make_budget_breach_args` to convert an over-budget event into the
structured terminator. The point: the agent has a *graceful* stop
signal that's auditable, not a kill -9 from the runtime.

```python
from arcrun.builtins import make_task_complete_tool, make_budget_breach_args
complete_tool = make_task_complete_tool()
# loop sees usage cross threshold → emits budget breach via task_complete
```

Combined with arcllm's `telemetry={'budget_scope': 'agent:plasma-007'}`
you get per-agent budgets enforced at the LLM boundary AND graceful
termination at the loop boundary. Belt and suspenders.

## 8. The full security stack on one call

What it looks like to compose them. Each module is opt-in via
`connect()` kwargs; the loop applies the sandbox and budget;
the audit chain (notebook 07) records every event.

```python
model = connect(
    security={'pii_enabled': True, 'signing_enabled': True},  # PII redact + Ed25519 sign
    audit={'log_level': 'INFO'},                              # PII-safe metadata logs
    telemetry={'budget_scope': 'agent:plasma-007'},           # cost ceilings
    rate_limit=True,                                          # provider TPS caps
    retry=True,                                               # exponential backoff
    fallback=True,                                            # provider failover
    otel=True,                                                # OpenTelemetry export
)

result = await run(
    model=model,
    tools=[make_execute_tool(timeout_seconds=10, max_output_bytes=8192)],
    sandbox=SandboxConfig(allowed_tools=['execute_python']),
    system_prompt=YOUR_SYSTEM,
    task=YOUR_TASK,
)

assert result.verify_integrity().valid  # tamper-evident chain
```

That's the full federal-tier posture. Every module composable,
every default sane, every event audited.

## Takeaway

| Defense | Mechanism | OWASP map |
|---|---|---|
| PII redaction | `security={'pii_enabled': True}` on `connect()` | LLM02 (Sensitive Disclosure) |
| Tool allowlist | `SandboxConfig(allowed_tools=[...])` | LLM06 (Excessive Agency) |
| Sandboxed code exec | `make_execute_tool` (subprocess) / `make_contained_execute_tool` (Docker) | LLM05 + ASI05 (RCE) |
| PII-safe audit | `audit={...}` (metadata-only by default) | LLM07 (log hygiene) |
| Cost / token budgets | `telemetry={'budget_scope': ...}` + `task_complete` terminator | LLM10 (Unbounded Consumption) |
| Tamper-evident chain | `verify_chain()` (notebook 07) | NIST AU-9, AU-10 |

**Each defense is one keyword argument away.** No code branches,
no separate codepath, no "federal mode fork." Same agent code; the
modules and the sandbox flip with config.

**Next:** [09 — The Coding Workflow](./09_coding_workflow.ipynb).
Now that you've seen the runtime defenses, see the development
workflow that ensures every feature ships with them turned on.